In [1]:
import argparse
import os
import re

import torch.nn.functional as F

os.environ["TORCHINDUCTOR_DISABLE"] = "1"
os.environ["TORCH_COMPILE"] = "0"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["DISABLE_TORCH_COMPILE"] = "1"
os.environ["TRANSFORMERS_NO_COMPILE"] = "1"
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from safetensors.torch import load_file as load_safetensors
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (adjusted_rand_score, balanced_accuracy_score,
                             calinski_harabasz_score, classification_report,
                             davies_bouldin_score, pairwise_distances,
                             silhouette_score)
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.model_selection import train_test_split
import pandas as pd
import seaborn as sns
import umap
import glob
from scipy.stats import spearmanr


/home/fe/purelku/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
/home/fe/purelku/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
models = [ "google/gemma-2-2b-it", "meta-llama/Llama-3.2-3B-Instruct", "google/gemma-2-2b", "meta-llama/Llama-3.2-3B",
           "Qwen/Qwen2.5-3B", "Qwen/Qwen2.5-3B-Instruct", "allenai/OLMo-2-0425-1B", "allenai/OLMo-2-0425-1B-Instruct"]


output_dir = "/data/erblina/Master_thesis"

datasets_list = ["walledai/AdvBench","walledai/DTStereotype", "walledai/CatHarmfulQA","walledai/DTToxicity","truthfulqa/truthful_qa"]


model_steering_1 = {'Qwen/Qwen2.5-3B': {'layers': [ 'model.layers.19', 'model.layers.20', 'model.layers.22'], 'alphas_up': [ 1.6, 1.6, 1.6], 'alphas_down': [ -1.8, -2.0, -2.0], 'max_avg_tox': [0.87, 0.87, 0.795, 0.76], 'min_avg_tox': [0.21, 0.21, 0.22, 0.19]},
    'Qwen/Qwen2.5-3B-Instruct': {'layers': [ 'model.layers.21', 'model.layers.20', 'model.layers.22'], 'alphas_up': [ 2.0, 2.0, 2.0], 'alphas_down': [ -0.6, -0.6, -0.6], 'max_avg_tox': [0.79, 0.79, 0.785, 0.78], 'min_avg_tox': [0.0, 0.0, 0.0, 0.0]},
    'allenai/OLMo-2-0425-1B-Instruct': {'layers': [ 'model.layers.9', 'model.layers.7', 'model.layers.8'], 'alphas_up': [ 2.0, 1.8, 1.6], 'alphas_down': [ -0.8, -1.0, -0.8], 'max_avg_tox': [0.75, 0.75, 0.735, 0.715], 'min_avg_tox': [0.0, 0.0, 0.0, 0.0]},
    'allenai/OLMo-2-0425-1B': {'layers': ['model.layers.5', 'model.layers.7', 'model.layers.4'], 'alphas_up': [-0.15, -0.07, 0.05], 'alphas_down': [-2.0, -1.5, -2.0], 'max_avg_tox': [0.4, 0.39, 0.39], 'min_avg_tox': [0.09, 0.085, 0.09]},
    'google/gemma-2-2b-it': {'layers': [ 'model.layers.10', 'model.layers.11', 'model.layers.12'], 'alphas_up': [ 1.5, 1.1, 1.0], 'alphas_down': [-0.3, -0.25, -0.2], 'max_avg_tox': [0.63, 0.63, 0.615, 0.595], 'min_avg_tox': [0.0, 0.0, 0.0, 0.0]},
    'meta-llama/Llama-3.2-3B-Instruct': {'layers': [ 'model.layers.12', 'model.layers.13', 'model.layers.14'], 'alphas_up': [ 2.0, 1.6, 2.0], 'alphas_down': [ -0.8, -0.5, -0.5], 'max_avg_tox': [0.82, 0.82, 0.81, 0.79], 'min_avg_tox': [0.0, 0.0, 0.0, 0.0]},
    'google/gemma-2-2b': {'layers': [ 'model.layers.6', 'model.layers.7', 'model.layers.13'], 'alphas_up': [ 1.2, 1.3, 1.4], 'alphas_down': [ -2.0, -1.8, -1.8], 'max_avg_tox': [0.36, 0.36, 0.35, 0.36], 'min_avg_tox': [0.135, 0.02, 0.035, 0.065]},
    'meta-llama/Llama-3.2-3B': {'layers': [ 'model.layers.12', 'model.layers.10', 'model.layers.11'], 'alphas_up': [ 0.7, 1.0, 1.0], 'alphas_down': [ -2.0, -1.4, -1.6], 'max_avg_tox': [0.605, 0.57, 0.575, 0.6], 'min_avg_tox': [0.385, 0.3, 0.33, 0.36]},
        }

In [3]:
def load_harmbench(model, output_dir):
    safe_model_name = re.sub(r'[\\/*?:"<>|]', "_", model)

    ###### HARMBENCH DATASET ######
    labels = np.load(f"{output_dir}/{safe_model_name}/labels.npy")
    hidden_states = load_safetensors(
        os.path.join(f"{output_dir}/{safe_model_name}", f"hidden_states_pure.safetensors"))
    
    side = 'toxic'
    # # Define path pattern
    # pattern = f"{output_dir}/{safe_model_name}/labels_steering_{side}_alpha_*.npy"

    # # Find all matching files
    # files = sorted(glob.glob(pattern))

    # # Load them into a dictionary keyed by alpha
    # data = {}
    # for file in files:
    #     # Extract the alpha value from filename
    #     alpha_str = os.path.splitext(os.path.basename(file))[0].split('_alpha_')[-1]
    #     alpha = float(alpha_str)
    #     data[alpha] = np.load(file, allow_pickle=True).item()

    return hidden_states, labels #, data

def load_datasets(model, dataset, output_dir):
    safe_model_name = re.sub(r'[\\/*?:"<>|]', "_", model)

    safe_dataset = re.sub(r'[\\/*?:"<>|]', "_", dataset)
    hidden_states_data = load_safetensors(os.path.join(output_dir, safe_model_name, f"{safe_dataset}_hidden_states_pure.safetensors"))
    labels_data = np.load(f"{output_dir}/{safe_model_name}/labels_{safe_dataset}.npy")

    return hidden_states_data, labels_data

def load_steering_dataset(model, dataset, model_steering_1, output_dir):
    side = 'toxic'
    safe_model_name = re.sub(r'[\\/*?:"<>|]', "_", model)
    safe_dataset = re.sub(r'[\\/*?:"<>|]', "_", dataset)
        # Find the layer corresponding to the given alpha
    layers = model_steering_1[model]['layers']
    data = {}
    for layer in layers:
        alpha = model_steering_1[model]['alphas_up'][layers.index(layer)] # or alphas_down
        labels_after_dataset = np.load(
                f"{output_dir}/{safe_model_name}/labels_steering_{side}_alpha_{alpha}_{safe_dataset}_{layer}.npy", allow_pickle=True).item()[layer]

        alpha_d = model_steering_1[model]['alphas_down'][layers.index(layer)]
        labels_after_dataset_d = np.load(
                f"{output_dir}/{safe_model_name}/labels_steering_{side}_alpha_{alpha_d}_{safe_dataset}_{layer}.npy", allow_pickle=True).item()[layer]

        data[layer] = {
            alpha : labels_after_dataset,
            alpha_d : labels_after_dataset_d
        }

    return data

def load_steering_harmbench(model, model_steering_1, output_dir):
    side = 'toxic'
    safe_model_name = re.sub(r'[\\/*?:"<>|]', "_", model)
        # Find the layer corresponding to the given alpha
    layers = model_steering_1[model]['layers']
    data = {}
    for layer in layers:
        alpha = model_steering_1[model]['alphas_up'][layers.index(layer)] # or alphas_down
        labels_after = np.load(
                f"{output_dir}/{safe_model_name}/labels_steering_{side}_alpha_{alpha}_{layer}.npy", allow_pickle=True).item()[layer]

        alpha_d = model_steering_1[model]['alphas_down'][layers.index(layer)]
        labels_after_d = np.load(
                f"{output_dir}/{safe_model_name}/labels_steering_{side}_alpha_{alpha_d}_{layer}.npy", allow_pickle=True).item()[layer]

        data[layer] = {
            alpha : labels_after,
            alpha_d : labels_after_d
        }

    return data


In [4]:
# Authors: The scikit-learn developers
# SPDX-License-Identifier: BSD-3-Clause

import matplotlib.pyplot as plt
import numpy as np

from sklearn import datasets
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, NeighborhoodComponentsAnalysis
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import umap.umap_ as umap  # <-- add this import


def plot_dim_reduction_comparison(X, y):
    n_neighbors = 5
    random_state = 0

    X_c = np.concatenate(X, axis=0)
    y_c = np.concatenate(y, axis=0)

    # Split into train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X_c, y_c, test_size=0.5, stratify=y_c, random_state=random_state
    )

    dim = len(X_c[0])
    n_classes = len(np.unique(y_c))

    # Reduce dimension to 2 with PCA
    pca = make_pipeline(StandardScaler(), PCA(n_components=2, random_state=random_state))

    # Reduce dimension to 2 with LinearDiscriminantAnalysis
    lda = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(n_components=2))

    # Reduce dimension to 2 with NeighborhoodComponentAnalysis
    nca = make_pipeline(
        StandardScaler(),
        NeighborhoodComponentsAnalysis(n_components=2, random_state=random_state),
    )

    # Use a nearest neighbor classifier to evaluate the methods
    knn = KNeighborsClassifier(n_neighbors=n_neighbors)

    # Make a list of the methods to be compared
    dim_reduction_methods = [("PCA", pca), ("NCA", nca)] #("LDA", lda)] #, 
    c = plt.cm.get_cmap("Set1", len(X))
    # plt.figure()
    for i, (name, model) in enumerate(dim_reduction_methods):
        plt.figure()
        # plt.subplot(1, 3, i + 1, aspect=1)

        # Fit the method's model
        model.fit(X_train, y_train)

        # Fit a nearest neighbor classifier on the embedded training set
        knn.fit(model.transform(X_train), y_train)

        # Compute the nearest neighbor accuracy on the embedded test set
        acc_knn = knn.score(model.transform(X_test), y_test)

        for i, (X_, y_) in enumerate(zip(X, y)):
            # Embed the data set in 2 dimensions using the fitted model
            X_embedded = model.transform(X_)

            # Plot the projected points and show the evaluation score
            plt.scatter(X_embedded[:, 0][y_ == 0], X_embedded[:, 1][y_ == 0], c=c(i), s=30, alpha=0.3, marker='o', zorder=1)

            plt.scatter(X_embedded[:, 0][y_ == 1], X_embedded[:, 1][y_ == 1], c=c(i), s=80, alpha=0.7, marker='x', zorder=1)

        plt.title(
            "{}, KNN (k={})\nTest accuracy = {:.2f}".format(name, n_neighbors, acc_knn)
        )
    plt.show()
    plt.close()


In [ ]:
for model in models: 
    layers = model_steering_1[model]['layers']
    plt.figure(figsize=(12,8))
    for i, layer in enumerate(layers):
        print(f"Processing model: {model}, layer: {layer}")
        X = []
        y = []
        X_1, v1 = load_harmbench(model, output_dir)
        print(sum(v1==1), sum(v1==0))
        v1 = (v1 > 0).astype(int)
        order1 = np.argsort(v1)  # zeros first, then ones

        v1 = v1[order1] # for balanced plotting
        idx1 = torch.from_numpy(order1).long()
        X_1[layer] = X_1[layer].index_select(dim=0, index=idx1).float() # X
        X_1[layer] = F.normalize(X_1[layer], p=2, dim=1)
        cos = cosine_similarity(X_1[layer].numpy())
        print(sum(v1==1), sum(v1==0))
        X.append(X_1[layer].float().numpy())

        # plt.imshow(cos, cmap='hot', interpolation='nearest')
        # plt.colorbar()
        # plt.title(f"Cosine Similarity for {model}")
        # plt.show()
        # plt.close()

        y.append(v1)
        print(f"Model: {model}, Hidden states shape: {X_1[layer].shape}, Labels shape: {v1.shape}")
        for data in datasets_list:
            X_2, v2 = load_datasets(model, data, output_dir)
            v2 = (v2 > 0).astype(int)
            order2 = np.argsort(v2)  # zeros first, then ones
            X.append(X_2[layer].float().numpy())
            y.append(v2)
            print(sum(v2==1), sum(v2==0))
            print(f"  Dataset: {data}, Hidden states shape: {X_2[layer].shape}, Labels shape: {v2.shape}")
            v2 = v2[order2] # for balanced plotting
            idx2 = torch.from_numpy(order2).long()
            X_2[layer] = X_2[layer].index_select(dim=0, index=idx2).float() # for balanced plotting
            X_2[layer] = F.normalize(X_2[layer], p=2, dim=1)
            cos = cosine_similarity(X_2[layer].numpy())
            print(sum(v2==1), sum(v2==0))

            # plt.imshow(cos, cmap='hot', interpolation='nearest')
            # plt.colorbar()
            # plt.title(f"Cosine Similarity {data}")
            # plt.show()
            # plt.close()

        X_all = np.concatenate(X, axis=0)
        X_c = cosine_similarity(X_all)
        y_all = np.concatenate(y, axis=0)
        cos_tox = X_c[y_all==1][:, y_all==1]
        cos_nontox = X_c[y_all==0][:, y_all==0]
        cos_between = X_c[y_all==1][:, y_all==0]

        data = [
            cos_tox.flatten(),
            cos_nontox.flatten(),
            cos_between.flatten()
            ]
        labels = ["Toxic", "Non-Toxic", "Between Classes"]
        colors = ["tab:orange", "tab:green", "tab:blue"]  # consistent colors

        plt.subplot(3, 3, i + 1)
        bp = plt.boxplot(
            data,
            labels=labels,
            patch_artist=True,  # allows coloring boxes
            widths=0.3,
            showfliers=False,   # hide the black dots (outliers)
        )

        # Apply custom colors
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            # patch.set_alpha(1)  # optional transparency

        # Optional: adjust median, whisker, and cap colors for readability
        for median in bp['medians']:
            median.set(color='black', linewidth=1.2)
        for whisker in bp['whiskers']:
            whisker.set(color='black', linewidth=1)
        for cap in bp['caps']:
            cap.set(color='black', linewidth=1)
        for mean in bp['means']:
            mean.set(color='black', marker='D', markeredgecolor='black')

        
        
        
        
        plt.title(f"{model}, {layer}")
        plt.ylabel("Cosine Similarity")

        plt.subplot(3, 3, i + 4)
        sns.kdeplot(cos_between.flatten(), color='tab:blue', fill=True, alpha=0.3)
        sns.kdeplot(cos_tox.flatten(), color='tab:orange', fill=True, alpha=0.3)
        sns.kdeplot(cos_nontox.flatten(), color='tab:green', fill=True, alpha=0.3)
        
        plt.ylabel("Density")
        plt.xlabel("Cosine Similarity")
    plt.legend(["Between Classes", "Toxic", "Non-Toxic"])
    # plt.ylim(0.7, 1.0)  # adjust based on your range
    plt.tight_layout()
    plt.show()
    plt.close()
    # plt.hist(cos_tox.flatten(), bins=50, alpha=0.3, color='tab:orange', density=True, label='Toxic')
    # plt.hist(cos_nontox.flatten(), bins=50, alpha=0.3, color='tab:green', density=True, label='Non-Toxic')
    # plt.hist(cos_between.flatten(), bins=50, alpha=0.3, color='tab:blue', density=True, label='Between Classes')

    # sns.kdeplot(cos_tox.flatten(), color='tab:orange')
    # sns.kdeplot(cos_nontox.flatten(), color='tab:green')
    # sns.kdeplot(cos_between.flatten(), color='tab:blue')
    # plt.axvline(np.median(cos_tox), color='tab:orange', linestyle='--', alpha=0.7)
    # plt.axvline(np.median(cos_nontox), color='tab:green', linestyle='--', alpha=0.7)
    # plt.axvline(np.median(cos_between), color='tab:blue', linestyle='--', alpha=0.7)


    # # sns.kdeplot(x=X_c[y_all==1][:, y_all==1].flatten())
    # # sns.kdeplot(x=X_c[y_all==0][:, y_all==0].flatten())
    # # sns.kdeplot(x=X_c[y_all==1][:, y_all==0].flatten())
    # plt.text(np.median(cos_tox), 5, f"{np.median(cos_tox):.2f}", color='tab:orange', ha='center')
    # plt.text(np.median(cos_nontox), 5, f"{np.median(cos_nontox):.2f}", color='tab:green', ha='center')
    # plt.text(np.median(cos_between), 5, f"{np.median(cos_between):.2f}", color='tab:blue', ha='center')

    # plt.title(f"Cosine Similarity Concatenated for model {model}")
    # plt.xlabel("Cosine Similarity")
    # plt.ylabel("Density")
    # plt.legend(["Toxic", "Non-Toxic", "Between Classes"])
    # plt.show()

    # plt.imshow(X_c, cmap='hot', interpolation='nearest')
    # plt.colorbar()
    # plt.title(f"Cosine Similarity Concatenated for model {model}")

    # model = umap.UMAP(n_neighbors=15, n_components=2, random_state=42, metric='cosine')
    # X_umap = model.fit(X_all)

    # plt.figure()
    # c = plt.cm.get_cmap("Set1", len(X))
    # for i, (X_, y_) in enumerate(zip(X, y)):
    #     # Embed the data set in 2 dimensions using the fitted model
    #     X_embedded = model.transform(X_)
    #     plt.subplot(3,3,i+1)
    #     # Plot the projected points and show the evaluation score
    #     plt.scatter(X_embedded[:, 0][y_ == 0], X_embedded[:, 1][y_ == 0], c=c(i), s=30, alpha=0.3, marker='o', zorder=1)

    #     plt.scatter(X_embedded[:, 0][y_ == 1], X_embedded[:, 1][y_ == 1], c=c(i), s=80, alpha=0.7, marker='x', zorder=2)
    # plt.title(f"UMAP Projection for model {model}")
    # plt.show()
    # plt.close()
    
    # print(f"Concatenated shape for model {model}: {X.shape}, Labels shape: {y.shape}")
    # plot_dim_reduction_comparison(X, y)


In [19]:
def upper_triangle(matrix):
    i, j = np.triu_indices_from(matrix, k=1)
    return matrix[i, j]


def rsa_similarity(model_rdm, true_rdm, method='spearman'):
    # vectorize upper triangles
    if model_rdm.ndim == 2:
        m = upper_triangle(model_rdm)
        t = upper_triangle(true_rdm)
    else:
        m = model_rdm
        t = true_rdm

    if method == 'spearman':
        corr, _ = spearmanr(m, t)
    elif method == 'pearson':
        from scipy.stats import pearsonr
        corr, _ = pearsonr(m, t)
    elif method == 'kendall':
        from scipy.stats import kendalltau
        corr, _ = kendalltau(m, t)
    elif method == 'mse':
        corr = -np.mean((m - t) ** 2)  # negative so higher is better
    
    elif method == 'cka':
        x = m
        y = t
        x -= x.mean()
        y -= y.mean()
        corr = np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))
    else:
        raise ValueError("Method must be one of: spearman, pearson, kendall, mse, cka")

    return corr

In [14]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d  # optional smoothing

def plot_metric_ribbon(res, layers, metric, dataset_names=None,
                       color="tab:blue", smooth=0, ci="sem",
                       show_individual=True):
    """
    res[layer][metric] = list of similarities per dataset (same order across layers)
    layers: list of layer keys (sorted)
    dataset_names: list of dataset names in the same order as the list in res[layer][metric]
    ci: "sem" for mean±1.96*SEM, "std" for mean±STD
    smooth: moving-average window (0/1 = no smoothing)
    show_individual: plot thin lines per dataset (gray) for context
    """
    # Build matrix: D x L (datasets × layers)
    sims = np.array([res[layer][metric] for layer in layers]).T  # shape (D, L)

    # If NaNs exist, use nan-safe stats
    mean = np.nanmean(sims, axis=0)
    if ci == "sem":
        sem = np.nanstd(sims, axis=0, ddof=1) / np.sqrt(np.sum(~np.isnan(sims), axis=0))
        lo, hi = mean - 1.96 * sem, mean + 1.96 * sem
    else:  # "std"
        std = np.nanstd(sims, axis=0, ddof=1)
        lo, hi = mean - std, mean + std

    # Optional smoothing for presentation
    if smooth and smooth > 1:
        mean = uniform_filter1d(mean, size=smooth, mode="nearest")
        lo   = uniform_filter1d(lo,   size=smooth, mode="nearest")
        hi   = uniform_filter1d(hi,   size=smooth, mode="nearest")

    # X-axis as numeric layer IDs if your keys are like "model.layers.12"
    try:
        layer_ids = [int(str(l).split('.')[-1]) for l in layers]
    except Exception:
        layer_ids = list(range(len(layers)))

    # Individual dataset lines (gray)
    if show_individual:
        for d in range(sims.shape[0]):
            plt.plot(layer_ids, sims[d], color="0.8", linewidth=1, alpha=0.7)

    # Ribbon + mean
    plt.fill_between(layer_ids, lo, hi, color=color, alpha=0.25, linewidth=0)
    plt.plot(layer_ids, mean, color=color, linewidth=2.5, label=f"{metric} mean ± {ci.upper()}")


In [ ]:
for model in models: 
    layers = model_steering_1[model]['layers']
    X_1, v1 = load_harmbench(model, output_dir)
    layers = list(X_1.keys())
    layers = sorted(layers, key=lambda x: int(x.split('.')[-1]))

    res = {}
    for i, layer in enumerate(layers):
        res[layer] = {}
        print(f"Processing model: {model}, layer: {layer}")
        X = []
        y = []
        cos_x = []
        cos_y = []
        
        print(sum(v1==1), sum(v1==0))
        v1 = (v1 > 0).astype(int)

        same_class = v1[:, None] == v1[None, :]
        # Dissimilarity = 1 - similarity
        cos_harmbench = 1 - cosine_similarity(X_1[layer].float().numpy())

        harmbench_true = 1 - same_class.astype(float)
        for metric in ['spearman', 'pearson', 'kendall',  'cka']: #'mse',
            sim = rsa_similarity(cos_harmbench, harmbench_true, method=metric)
            res[layer][metric] = [sim]
            # print(f"  Harmbench, Metric: {metric}, RSA similarity: {sim:.4f}")

        X.append(X_1[layer].float().numpy())
        y.append(v1)
        print(f"Model: {model}, Hidden states shape: {X_1[layer].shape}, Labels shape: {v1.shape}")
        for data in datasets_list:
            X_2, v2 = load_datasets(model, data, output_dir)
            v2 = (v2 > 0).astype(int)

            same_class = v2[:, None] == v2[None, :]
            # Dissimilarity = 1 - similarity
            
            cos_harmbench = 1 - cosine_similarity(X_2[layer].float().numpy())
            harmbench_true = 1 - same_class.astype(float)

            for metric in ['spearman', 'pearson', 'kendall',  'cka']: #'mse',
                sim = rsa_similarity(cos_harmbench, harmbench_true, method=metric)
                res[layer][metric].append(sim)
                # print(f"  Dataset: {data}, Metric: {metric}, RSA similarity: {sim:.4f}")

            X.append(X_2[layer].float().numpy())
            y.append(v2)
            

        # X_all = np.concatenate(X, axis=0)
        # X_c = 1 -cosine_similarity(X_all)
        # y_all = np.concatenate(y, axis=0)

        # same_class = y_all[:, None] == y_all[None, :]
        # Dissimilarity = 1 - similarity

        # y_c = 1 - same_class.astype(float)

        # for metric in ['spearman', 'pearson', 'kendall',  'cka']: #'mse',
        #     sim = rsa_similarity(cos_harmbench, harmbench_true, method=metric)
        #     res[layer][f"harmbench_{metric}"] = sim
        #     sim_all = rsa_similarity(X_c, y_c, method=metric)
        #     res[layer][f"all_datasets_{metric}"] = sim_all
        #     print(f"  Datasets all, Metric: {metric}, RSA similarity: {sim_all:.4f}")

    os.makedirs("/home/fe/purelku/Desktop/Master_thesis/results_RSA/", exist_ok=True)
    safe_model_name = re.sub(r'[\\/*?:"<>|]', "_", model)
    plt.figure(figsize=(8, 6))
    for i, m in enumerate(['spearman', 'pearson', 'kendall',  'cka']):
        plt.subplot(2, 2, i+1)
        plot_metric_ribbon(res, layers, metric=m, color=["tab:blue","tab:orange","tab:green","tab:purple"][i],
                        smooth=2, ci="sem", show_individual=True)
        plt.xlabel("Layer")
        plt.ylabel("RSA similarity")
        plt.title(f"Metric: {m.upper()}")
        plt.grid(alpha=0.3)
        # plt.legend(loc="lower right")

    plt.suptitle(f'RSA Similarity {model}')

    plt.tight_layout()
    plt.savefig(f"/home/fe/purelku/Desktop/Master_thesis/results_RSA/{safe_model_name}_rsa_similarity_ribbon.png", dpi=300, bbox_inches='tight')
    # plt.show()
    # plt.show()
    plt.close()
    
    # plt.figure(figsize=(14, 12))

    # for i, method in enumerate(['spearman', 'pearson', 'kendall',  'cka']): #'mse',
    #     plt.subplot(2, 2, i+1)

    #     harmbench_vals = [res[layer][method][0] for layer in layers]
    #     plt.plot(layers, harmbench_vals, marker='o', label='Harmbench')
    #     for data in range(1, len(datasets_list)+1):
    #         all_datasets_vals = [res[layer][method][data] for layer in layers]
    #         plt.plot(layers, all_datasets_vals, marker='o', label=datasets_list[data-1].split('/')[-1])

    #     plt.xticks(rotation=45)
    #     plt.xlabel('Layers')
    #     plt.ylabel('RSA Similarity')
    #     plt.title(f'Metric: {method}')
    
    # plt.legend()
    # plt.suptitle(f'RSA Similarity {model}')
    # plt.tight_layout()
    # plt.savefig(f"/home/fe/purelku/Desktop/Master_thesis/results_RSA/{safe_model_name}_rsa_similarity.png", dpi=300, bbox_inches='tight')
    # # plt.show()

    # plt.close()

Processing model: google/gemma-2-2b-it, layer: model.layers.0
2 198
Model: google/gemma-2-2b-it, Hidden states shape: torch.Size([200, 2304]), Labels shape: (200,)
Processing model: google/gemma-2-2b-it, layer: model.layers.1
2 198
Model: google/gemma-2-2b-it, Hidden states shape: torch.Size([200, 2304]), Labels shape: (200,)
Processing model: google/gemma-2-2b-it, layer: model.layers.2
2 198
Model: google/gemma-2-2b-it, Hidden states shape: torch.Size([200, 2304]), Labels shape: (200,)
Processing model: google/gemma-2-2b-it, layer: model.layers.3
2 198
Model: google/gemma-2-2b-it, Hidden states shape: torch.Size([200, 2304]), Labels shape: (200,)
Processing model: google/gemma-2-2b-it, layer: model.layers.4
2 198
Model: google/gemma-2-2b-it, Hidden states shape: torch.Size([200, 2304]), Labels shape: (200,)
Processing model: google/gemma-2-2b-it, layer: model.layers.5
2 198
Model: google/gemma-2-2b-it, Hidden states shape: torch.Size([200, 2304]), Labels shape: (200,)
Processing model

In [22]:
for model in models: 
    # layers = model_steering_1[model]['layers']
    safe_model_name = re.sub(r'[\\/*?:"<>|]', "_", model)

    X_1, v1 = load_harmbench(model, output_dir)
    layers = list(X_1.keys())
    layers = sorted(layers, key=lambda x: int(x.split('.')[-1]))
    t = 'nontoxic'
    res = {}
    steering_vector = torch.load(os.path.join(f"{output_dir}/{safe_model_name}", "steering_vectors.pt"))
    for i, layer in enumerate(layers):
        res[layer] = {}
        print(f"Processing model: {model}, layer: {layer}")
        X = []
        y = []
        cos_x = []
        cos_y = []
        t_vec = steering_vector[layer][t].unsqueeze(0).float().numpy()  # shape (1, hidden_size)
        print(sum(v1==1), sum(v1==0))
        v1 = (v1 > 0).astype(int)

        # same_class = v1[:, None] == v1[None, :]
        # Dissimilarity = 1 - similarity
        cos_harmbench = 1 - cosine_similarity(X_1[layer].float().numpy(), t_vec).squeeze()
        if t == 'toxic':
            harmbench_true = 1 - v1.astype(float)
        else:
            harmbench_true = v1.astype(float)


        for metric in ['spearman', 'pearson', 'kendall',  'cka']: #'mse',
            sim = rsa_similarity(cos_harmbench, harmbench_true, method=metric)
            res[layer][metric] = [sim]
            # print(f"  Harmbench, Metric: {metric}, RSA similarity: {sim:.4f}")

        X.append(X_1[layer].float().numpy())
        y.append(v1)
        # print(f"Model: {model}, Hidden states shape: {X_1[layer].shape}, Labels shape: {v1.shape}")
        for data in datasets_list:
            X_2, v2 = load_datasets(model, data, output_dir)
            v2 = (v2 > 0).astype(int)

            # Dissimilarity = 1 - similarity

            cos_harmbench = 1 - cosine_similarity(X_2[layer].float().numpy(), t_vec).squeeze()
            if t == 'toxic':
                harmbench_true = 1 - v2.astype(float)
            else:
                harmbench_true = v2.astype(float)

            for metric in ['spearman', 'pearson', 'kendall',  'cka']: #'mse',
                sim = rsa_similarity(cos_harmbench, harmbench_true, method=metric)
                res[layer][metric].append(sim)
                # print(f"  Dataset: {data}, Metric: {metric}, RSA similarity: {sim:.4f}")

            X.append(X_2[layer].float().numpy())
            y.append(v2)
            


    os.makedirs("/home/fe/purelku/Desktop/Master_thesis/results_RSA/", exist_ok=True)
    plt.figure(figsize=(8, 6))
    for i, m in enumerate(['spearman', 'pearson', 'kendall',  'cka']):
        plt.subplot(2, 2, i+1)
        plot_metric_ribbon(res, layers, metric=m, color=["tab:blue","tab:orange","tab:green","tab:purple"][i],
                        smooth=2, ci="sem", show_individual=True)
        plt.xlabel("Layer")
        plt.ylabel("RSA similarity")
        plt.title(f"Metric: {m.upper()}")
        plt.grid(alpha=0.3)
        # plt.legend(loc="lower right")

    plt.suptitle(f'RSA Similarity with {t} sv {model}')

    plt.tight_layout()
    plt.savefig(f"/home/fe/purelku/Desktop/Master_thesis/results_RSA/{safe_model_name}_rsa_similarity_ribbon_{t}_sv.png", dpi=300, bbox_inches='tight')
    # plt.show()
    # plt.show()
    plt.close()
    

Processing model: google/gemma-2-2b-it, layer: model.layers.0
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.1
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.2
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.3
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.4
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.5
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.6
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.7
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.8
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.9
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.10
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.11
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.12
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.13
2 198
Processing model: google/gemma-2-2b-it, laye

In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
for model in models: 
    # layers = model_steering_1[model]['layers']
    safe_model_name = re.sub(r'[\\/*?:"<>|]', "_", model)

    X_1, v1 = load_harmbench(model, output_dir)
    layers = list(X_1.keys())
    layers = sorted(layers, key=lambda x: int(x.split('.')[-1]))
    t = 'toxic'
    res = {}
    steering_vector = torch.load(os.path.join(f"{output_dir}/{safe_model_name}", "steering_vectors.pt"))
    for i, layer in enumerate(layers):
        res[layer] = {}
        print(f"Processing model: {model}, layer: {layer}")
        X = []
        y = []
        cos_x = []
        cos_y = []
        t_vec = steering_vector[layer][t].float().numpy()  # shape (1, hidden_size)
        print(sum(v1==1), sum(v1==0))
        v1 = (v1 > 0).astype(int)

        proj = X_1[layer].float().numpy() @ t_vec
        clf = LogisticRegression(class_weight='balanced', solver='liblinear').fit(proj.reshape(-1, 1), v1)
        preds = clf.predict_proba(proj.reshape(-1, 1))[:, 1]
        auc = roc_auc_score(v1, preds)
        res[layer]['auc'] = [auc]


        X.append(X_1[layer].float().numpy())
        y.append(v1)
        # print(f"Model: {model}, Hidden states shape: {X_1[layer].shape}, Labels shape: {v1.shape}")
        for data in datasets_list:
            X_2, v2 = load_datasets(model, data, output_dir)
            v2 = (v2 > 0).astype(int)

            proj = X_2[layer].float().numpy() @ t_vec
            # clf = LogisticRegression(class_weight='balanced', solver='liblinear').fit(proj.reshape(-1, 1), v2)
            preds = clf.predict_proba(proj.reshape(-1, 1))[:, 1]
            auc = roc_auc_score(v2, preds)
            res[layer]['auc'].append(auc)

            

            X.append(X_2[layer].float().numpy())
            y.append(v2)
            


    os.makedirs("/home/fe/purelku/Desktop/Master_thesis/results_projection_analysis/", exist_ok=True)
    plt.figure(figsize=(10, 4))
    for i, m in enumerate(['auc']): #'spearman', 'pearson', 'kendall',  'cka']):
        plt.subplot(1, 2, 1)
        plot_metric_ribbon(res, layers, metric=m, color=["tab:blue","tab:orange","tab:green","tab:purple"][i],
                        smooth=2, ci="sem", show_individual=True)
                
        plt.xlabel("Layer")
        plt.ylabel("AUC")
        plt.title(f"Metric: {m.upper()}")
        plt.subplot(1, 2, 2)
        for j, d in enumerate(['w/Harmbench'] + datasets_list):
            aucs = [res[layer][m][j] for layer in layers]
            plt.plot([int(l.split('.')[-1]) for l in layers], aucs, marker='.', label=d.split('/')[-1])
                
        plt.xlabel("Layer")
        plt.ylabel("AUC")
        plt.title(f"Metric: {m.upper()}")
        # plt.grid(alpha=0.3)
        plt.legend(loc="upper right", bbox_to_anchor=(1.5, 1.05))
        

    plt.suptitle(f'Projection with {t} sv {model}')

    plt.tight_layout()
    plt.savefig(f"/home/fe/purelku/Desktop/Master_thesis/results_projection_analysis/{safe_model_name}_auc_{t}_sv.png", dpi=300, bbox_inches='tight')
    # plt.show()
    # plt.show()
    plt.close()
    



Processing model: google/gemma-2-2b-it, layer: model.layers.0
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.1
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.2
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.3
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.4
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.5
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.6
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.7
2 198


Processing model: google/gemma-2-2b-it, layer: model.layers.8
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.9
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.10
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.11
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.12
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.13
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.14
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.15
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.16
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.17
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.18
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.19
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.20
2 198
Processing model: google/gemma-2-2b-it, layer: model.layers.21
2 198
Processing model: google/gemma-2-2b-

In [46]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mutual_info_score
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist, squareform

# --- Optional: CKA helper ---
def compute_cka(X, Y):
    """Centered Kernel Alignment between two representation matrices."""
    X = X - X.mean(0, keepdims=True)
    Y = Y - Y.mean(0, keepdims=True)
    Kx = X @ X.T
    Ky = Y @ Y.T
    hsic = np.sum(Kx * Ky)
    norm = np.sqrt(np.sum(Kx * Kx) * np.sum(Ky * Ky))
    return hsic / norm


# --- Helper for RSA (C) ---
def upper_triangle(mat):
    return mat[np.triu_indices_from(mat, k=1)]


def rsa_rdm(model_rdm, true_rdm):
    """RSA correlation (Spearman) between model and true categorical RDMs."""
    m = upper_triangle(model_rdm)
    t = upper_triangle(true_rdm)
    corr, _ = spearmanr(m, t)
    return corr


# --- Start analysis ---
for model in models:
    safe_model_name = re.sub(r'[\\/*?:"<>|]', "_", model)
    X_1, v1 = load_harmbench(model, output_dir)
    layers = sorted(list(X_1.keys()), key=lambda x: int(x.split('.')[-1]))

    print(f"\n=== Processing model: {model} ===")
    results = {} #{'Δ': [], 'RSA': [], 'CKA': [], 'MI': []}

    for layer in layers:
        X = X_1[layer].float().numpy()
        y = (v1 > 0).astype(int)
        results[layer] = {}

        # --- B. Contrastive Δ (mean distance difference) ---
        cos = cosine_similarity(X)
        dist_to_class_mean = []
        for c in [0, 1]:
            cls_vecs = X[y == c].mean(0, keepdims=True)
            dists = 1 - cosine_similarity(X, cls_vecs).squeeze()
            dist_to_class_mean.append(dists)
        Δ = np.mean(dist_to_class_mean[1]) - np.mean(dist_to_class_mean[0]) 
        results[layer]['Δ'] = [Δ]

        # --- C. RSA (model RDM vs. true categorical RDM) ---
        model_rdm = 1 - cosine_similarity(X)
        true_rdm = 1 - (y[:, None] == y[None, :]).astype(float)
        rsa_val = rsa_rdm(model_rdm, true_rdm)
        results[layer]['RSA'] = [rsa_val]

        # --- D. CKA (representational alignment with categorical kernel) ---
        y_vec = y.reshape(-1, 1)
        cat_kernel = y_vec @ y_vec.T  # binary kernel
        cka_val = compute_cka(X, cat_kernel)
        results[layer]['CKA'] = [cka_val]

        # --- E. Mutual Information (representation vs. label) ---
        # Flatten activations along feature axis for MI
        proj = X @ np.random.randn(X.shape[1], 1)  # random projection for stability
        proj_disc = np.digitize(proj.squeeze(), np.histogram(proj, bins=20)[1])
        mi_val = mutual_info_score(proj_disc, y)
        results[layer]['MI'] = [mi_val]

        for data in datasets_list:
            X_2, v2 = load_datasets(model, data, output_dir)
            v2 = (v2 > 0).astype(int)
            X = X_2[layer].float().numpy()
            y = v2

            # --- B. Contrastive Δ (mean distance difference) ---
            cos = cosine_similarity(X)
            dist_to_class_mean = []
            for c in [0, 1]:
                cls_vecs = X[y == c].mean(0, keepdims=True)
                dists = 1 - cosine_similarity(X, cls_vecs).squeeze()
                dist_to_class_mean.append(dists)
            Δ = np.mean(dist_to_class_mean[1]) - np.mean(dist_to_class_mean[0]) 
            results[layer]['Δ'].append(Δ)

            # --- C. RSA (model RDM vs. true categorical RDM) ---
            model_rdm = 1 - cosine_similarity(X)
            true_rdm = 1 - (y[:, None] == y[None, :]).astype(float)
            rsa_val = rsa_rdm(model_rdm, true_rdm)
            results[layer]['RSA'].append(rsa_val)

            # --- D. CKA (representational alignment with categorical kernel) ---
            y_vec = y.reshape(-1, 1)
            cat_kernel = y_vec @ y_vec.T  # binary kernel
            cka_val = compute_cka(X, cat_kernel)
            results[layer]['CKA'].append(cka_val)

            # --- E. Mutual Information (representation vs. label) ---
            # Flatten activations along feature axis for MI
            proj = X @ np.random.randn(X.shape[1], 1)  # random projection for stability
            proj_disc = np.digitize(proj.squeeze(), np.histogram(proj, bins=20)[1])
            mi_val = mutual_info_score(proj_disc, y)
            results[layer]['MI'].append(mi_val)

    # --- Plotting ---
    fig, axs = plt.subplots(2, 2, figsize=(10, 8))
    metrics = ['Δ', 'RSA', 'CKA', 'MI']
    titles = [
        'Δ (Contrastive Distance Difference (Toxic - Nontoxic))',
        'RSA (RDM vs Categorical RDM)',
        'CKA (Representational Alignment)',
        'Mutual Information (Representation vs Label)'
    ]
    for i, metric in enumerate(metrics):
        ax = axs.flat[i]
        for d, data in enumerate(['y/Harmbench'] + datasets_list):
            data_vals = [results[layer][metric][d] for layer in layers]

            ax.plot([int(l.split('.')[-1]) for l in layers], data_vals, marker='.', label=data.split('/')[-1])
        # ax.plot(layers, results[metric], marker='o', linewidth=2)
        ax.set_title(titles[i])
        ax.set_xlabel("Layer")
        ax.set_ylabel(metric)
        ax.grid(alpha=0.3)
        ax.tick_params(axis='x', rotation=45)
    plt.legend(loc="upper right", bbox_to_anchor=(1.5, 1.05))
    plt.suptitle(f"{model} — Representational Metrics", fontsize=13)
    plt.tight_layout()
    # plt.show()
    plt.savefig(f"/home/fe/purelku/Desktop/Master_thesis/results_RSA/{safe_model_name}_rsa_all_metrics.png",
                dpi=300, bbox_inches='tight')
    plt.close()



=== Processing model: google/gemma-2-2b-it ===

=== Processing model: meta-llama/Llama-3.2-3B-Instruct ===

=== Processing model: google/gemma-2-2b ===

=== Processing model: meta-llama/Llama-3.2-3B ===

=== Processing model: Qwen/Qwen2.5-3B ===

=== Processing model: Qwen/Qwen2.5-3B-Instruct ===

=== Processing model: allenai/OLMo-2-0425-1B ===

=== Processing model: allenai/OLMo-2-0425-1B-Instruct ===
